#  LLM Flake Analysis Automation: Step 2 (alternative version)

---

### Goal: Use database of flakes to automate optimal flake selection based on user input


**By: Sanjit Masanam (2025, UCSB)**

## Setup

### **NOTE: "Setup" section is the same as Step 1 (alt). Only "The Fun Part" differs.**

Here we'll do some setup actions such as mounting the Lab Google Drive to access LLM models + flake images. When prompted, click "Continue" twice to give Google Colab access to the Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
main_path = '/content/drive/Shareddrives/Yankowitz Lab/'

Mounted at /content/drive


We'll need to install the following requirements:
- llama-stack

In [ ]:
!pip install llama-stack

We'll also need to download the model ([LLama-3.2-11B-Vision](https://huggingface.co/meta-llama/Llama-3.2-11B-Vision)) from HuggingFace in case the local download is incomplete or not present for any reason. If the local model has no issues, this step should only a few seconds.

If the hf_token no longer works, visit [here](https://huggingface.co/docs/hub/en/security-tokens).

Note: Current access token belongs to Sanjit Masanam so setting up one for the Lab could be useful.

Note 2: One common issue with the download is that the entire model isn't saved to the Drive. If this occurs, try the download again after restarting the runtime. Once the model is fully downloaded, this issue shouldn't arise again (🤞).  

In [ ]:
from huggingface_hub.hf_api import HfFolder
# HfFolder.save_token('') ## Set access token

In [ ]:
from huggingface_hub import snapshot_download
import os

local_path = '/content/drive/Shareddrives/Yankowitz Lab/LLM/Llama-3.2-11B-Vision'
os.makedirs(local_path, exist_ok = True)

snapshot_download(repo_id="meta-llama/Llama-3.2-11B-Vision", repo_type="model", local_dir=local_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

LICENSE.txt:   0%|          | 0.00/7.71k [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

USE_POLICY.md:   0%|          | 0.00/6.02k [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/459 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.7k [00:00<?, ?B/s]

KeyboardInterrupt: 

## Fun Part

Now... it's time. Run the code below and follow the prompts to have an LLM analyze your exfoliated flake(s)!

**Status (Aug. 2025)**: Since Step 1 is still a work in progress, Step 2 cannot be tested

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
import os

# ------------------------------------------------------------------------------------------------------------------------
def initializeLLM(LLM_path):
    processor = AutoProcessor.from_pretrained(LLM_path)
    model = AutoModelForVision2Seq.from_pretrained(LLM_path)

    return processor, model
# ------------------------------------------------------------------------------------------------------------------------

### Initialize dataframe and model
flake_df = pd.read_csv('flake_df.csv')
print(flake_df.head)

print("What stack do you want to make?")
overall_stack = input()
print("Please list each layer of your stack in a comma seperated list (layer_1, layer_2, etc.)")
stack_layers = input()
print("Please provide any extra information about your desired stack (optional)")
misc_info = input()

dataset_dict = flake_df.to_dict('records')

prompt = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": f"{dataset_dict}. This is a dataset of 2D vdW flakes I have. I want to make a 2D van der Waals {overall_stack}. The layers are {stack_layers}. {misc_info}. Please provide 3 potential stacks I could make based on the category ratings and tags. For each potential stack, please output a comma seperated list of the flake_filepath for each flake and end the list with a short description of that potential stack."}
        ]
    },
]

processor, model = initializeLLM(local_path)

inputs = processor.apply_chat_template(
	prompt,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Hello! This is LLMatt at your service (most of the time...)! Please enter the full path to flake or directory of the flakes you would like me to analyze:
/content/drive/Shareddrives/Yankowitz Lab/People/Sanjit M/Exfoliations/SM_hBN_001/Chip 5/f1_100x_cs.jpg
What material is this flake(s) made of:
hBN
Should I only analyze files with a certain naming convention? If so, please 
Thanks, I'll work on analyzing now!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

KeyboardInterrupt: 